In [1]:
from pathlib import Path
import cv2
import numpy as np
from mmdet.apis import inference_detector, init_detector

In [2]:
config_path = Path(r"C:\dev\projects\CV_counting_bags\configs\rtmdet_tiny_bag.py")
ckpt_path = Path(r"C:\dev\projects\CV_counting_bags\work_dirs\rtmdet_tiny_bag\best_coco_bbox_mAP_epoch_27.pth")
video_path = Path(r"C:\dev\projects\CV_counting_bags\input.mp4")
out_video_path = Path(r"C:\dev\projects\CV_counting_bags\output_video\output.mp4")

In [3]:
model = init_detector(
    str(config_path),
    str(ckpt_path),
    device = "cuda:0"
)

Loads checkpoint by local backend from path: C:\dev\projects\CV_counting_bags\work_dirs\rtmdet_tiny_bag\best_coco_bbox_mAP_epoch_27.pth


In [4]:
conveyor_roi = np.array([
    [0, 360],
    [310, 28],
    [484, 65],
    [315, 360],    
], dtype=np.int32)

In [14]:
def inside_roi(obj, roi):
    obj_center = (int((obj[0] + obj[2]) / 2), int((obj[1] + obj[3]) / 2))
    result = cv2.pointPolygonTest(roi, obj_center, measureDist = False)

    if result >= 0:
        return True

In [18]:
def draw_detection(frame, result, roi, threshold = 0.36):
    image = frame.copy()

    pred = result.pred_instances
    bboxes = pred.bboxes.detach().cpu().numpy()
    scores = pred.scores.detach().cpu().numpy()

    detections_sum = 0
    track_detections = []

    for bbox, score in zip(bboxes, scores):
        if score < threshold:
            continue

        if inside_roi(bbox, roi):
            track_detections.append({
                "bbox": bbox,
                "score": score
            })

        detections_sum += 1

        x1, y1, x2, y2 = bbox.astype(int)

        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        label = f"bag {score:.2f}"

        cv2.putText(image, label, (x1, max(y1 - 7, 20)), 
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.putText(image, f"Detections: {detections_sum}",
        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    cv2.putText(image, f"Tracking candidates: {len(track_detections)}",
            (20, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    return image, track_detections

In [19]:
def process_video(viedo_path, out_path, model, threshold = 0.36, max_frames = None):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError("failed to open video:", video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frames_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"FPS: {fps}")
    print(f"Res: {width}x{height}")
    print(f"Frames: {frames_count}")

    out_path = Path(out_path)
    out_path.parent.mkdir(parents = True, exist_ok = True)
    
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (width, height))

    if not writer.isOpened():
        cap.release()
        raise RuntimeError("failed to create video", out_path)

    frame_index = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if max_frames is not None and frame_index >= max_frames:
            break

        result = inference_detector(model, frame)
        annotation, track_detections = draw_detection(frame, result, conveyor_roi)

        writer.write(annotation)
        frame_index += 1

    cap.release()
    writer.release()
    print("done")

In [21]:
#process_video(video_path, out_path = out_video_path, model = model, max_frames = 1500)

In [22]:
def bbox_iou(bbox1, bbox2):
    x1 = max(bbox1[0], bbox2[0])
    y1 = max(bbox1[1], bbox2[1])
    x2 = min(bbox1[2], bbox2[2])
    y2 = min(bbox1[3], bbox2[3])

    inter_width = max(0, x2 - x1)
    inter_height = max(0, y2 - y1)

    area1 = max(0, bbox1[2] - bbox1[0]) * max(0, bbox1[3] - bbox1[1])
    area2 = max(0, bbox2[2] - bbox2[0]) * max(0, bbox2[3] - bbox2[1])

    intersection = inter_width * inter_height
    union = area1 + area2 - intersection
    
    if union == 0:
        return 0.0

    return intersection / union

In [23]:
class Track:
    def __init__(self, track_id, bbox, score):
        self.id = track_id
        self.bbox = np.array(bbox, dtype = float)
        self.score = float(score)

        self.hits = 1
        self.missed = 0
        self.age = 1
        self.confirmed = False
        self.history = [self.center()]

    def center(self):
        x1, y1, x2, y2 = self.bbox

        return(float((x1 + x2) / 2), float((y1 + y2) / 2))

    def update(self, bbox, score):
        self.bbox = np.array(bbox, dtype = float)
        self.score = float(score)

        self.hits += 1
        self.missed = 0
        self.age += 1
        self.history.append(self.center())

    def mark_missed(self):
        self.missed += 1
        self.age += 1